# Breakout 4B · Sprint 1 · match the 25 importers to the tax register

Each importer as customs holds it, against the tax register's best candidate; the similarities are computed for you. **I17 is worked.** You compute the confidence and the class at the dial for all 25, then set the dial on Track B's labelled sample.

*Published customs datasets record an irregularity on 1 to 8 lines in 100; in this lab the random draw finds 3 in 100.*

In [ ]:
# Run this cell first (click it, then Shift + Enter). It finds the lab data.
import hashlib, os
from pathlib import Path
import numpy as np, pandas as pd

CANDIDATES = [os.environ.get('WORKSHOP_DATA', ''), 'data', '../data', '/workspace/data', '.']
DATA = next(Path(c) for c in CANDIDATES if c and (Path(c) / 'tariff.csv').exists())
print('Data folder:', DATA)

def _h(v, nd):
    return hashlib.sha256(f'{round(float(v), nd):.{nd}f}'.encode()).hexdigest()[:16]
def _hs(v):
    return hashlib.sha256(str(v).strip().lower().encode()).hexdigest()[:16]
def check(name, value, expected, nd=None):
    if callable(value):
        try:
            value = value()
        except Exception:
            value = ...
    try:
        ok = value is not ... and (_hs(value) == expected if nd is None else _h(value, nd) == expected)
    except Exception:
        ok = False
    shown = value if nd is None or value is ... else f'{value:,.{nd}f}'
    print(('✓ ' if ok else '✗ not yet: ') + name + ' = ' + str(shown))


In [ ]:
pairs = pd.read_csv(DATA / 'pairs_4b.csv')[['ref', 'exact_status', 'best_tin', 'name_sim', 'address_sim', 'director_sim', 'phone_match']]
W = {'name_sim': 0.4, 'address_sim': 0.3, 'director_sim': 0.3}   # the recipe's weights: a choice, say it
pairs.head()

## Worked: I17

A missing field is left out and the weights are rescaled over the fields present: a missing field is not a zero.

In [ ]:
r = pairs.set_index('ref').loc['I17']
present = [k for k in W if pd.notna(r[k])]
confidence = sum(W[k] * r[k] for k in present) / sum(W[k] for k in present)
print(round(confidence, 3), '→ review band (0.70 to 0.90): a person reads both records')

## YOUR TURN: the 25, and the dial

In [ ]:
# YOUR TURN: a confidence and a class for every row
pairs['confidence'] = ...
pairs['class'] = ...   # exact (exact_status == 'matched'), validated probable (≥ 0.9), review (≥ 0.7), no link
scores = pd.read_csv(DATA / 'match_scores.csv'); sample = pd.read_csv(DATA / 'trackB_labelled_sample.csv')
precision = ...   # at the dial 0.7: true links ÷ sample pairs scored at or above 0.7
recall = ...      # true links at or above 0.7 ÷ all true links in the sample

In [ ]:
E = {'I01': '3cae67a480a37535', 'I02': '3cae67a480a37535', 'I03': '3cae67a480a37535', 'I04': '3cae67a480a37535', 'I05': '3cae67a480a37535', 'I06': '3cae67a480a37535', 'I07': '3cae67a480a37535', 'I08': '1ef3b92a991cfa4c', 'I09': '90e380245f828c34', 'I10': '3cae67a480a37535', 'I11': '3cae67a480a37535', 'I12': '7d40da9b2c38c48b', 'I13': '799d249a56de4ee3', 'I14': '3cae67a480a37535', 'I15': '3cae67a480a37535', 'I16': '3cae67a480a37535', 'I17': '8d66bd6ebaa6ec94', 'I18': '3cae67a480a37535', 'I19': '3cae67a480a37535', 'I20': '3cae67a480a37535', 'I21': '3cae67a480a37535', 'I22': '3cae67a480a37535', 'I23': '3cae67a480a37535', 'I24': '3cae67a480a37535', 'I25': '3cae67a480a37535'}
check('rows in the review band', lambda: (pairs['class'] == 'review').sum(), '4e07408562bedb8b', 0)
check('rows with no link', lambda: (pairs['class'] == 'no link').sum(), 'd4735e3a265e16ee', 0)
check('rows with the right confidence (of 25)', lambda: sum(_h(c, 3) == E[r] for r, c in pairs.set_index('ref').confidence.items() if r in E), 'b7a56873cd771f2c', 0)
check('Track B precision at 0.7', lambda: precision, '6270a154bf299d8a', 3)
check('Track B recall at 0.7', lambda: recall, '957b4675a7db67e8', 3)